# LogVar2FJ 4 — the forward-skew reserve, and what the priors are worth in a mark

Notebook 2 established that a vanilla-only ladder does not identify the residual's tail, and that
nothing in the market data available today closes it. A model that cannot see the forward smile
is a model that cannot mark a deal whose value depends on it — and an autocall's value depends on
it, because the deal is a sequence of conditional payoffs each of which is an option on the
spot's law at a later date.

So where the forward smile is not quoted, the number is a **reserve** rather than a mark. It is
built in two halves that meet at valuation time.

**The calibration's half** is written onto the factor: `Skew_Gradient`, the pair
`d(Delta_skew)/dBeta` and `d(Delta_skew)/dRho_S` in vol points per unit, taken in the last bucket
at the nearest forward tenor; and `Stickiness_Band`, the width in vol points that the forward
skew is uncertain by. The band is the spread between the two views a desk holds — sticky-delta
and a local-stochastic-vol-like one — and it defaults to half a vol point.

**The valuation's half** is the deal's own `dPV/dBeta` and `dPV/dRho_S`, which arrive on the tape
with the rest of the first-order greeks, because the residual's drift is forced rather than
fitted and so every parameter is reachable by the backward.

**The composition.** Two parameters carry one target, so the parameter move behind a vol point of
forward skew is not unique; the one taken is the **minimum-norm** move, `J' / (J J')`, which is
the same contraction the quote sensitivities take over their null space. The reserve is then
`|dPV/d(Delta_skew)| x band`, and it is reported as `Skew_Reserve` beside `Value`.

In [1]:
import copy, io, json, logging, os, sys, time

HERE = os.path.abspath(os.getcwd())
REPO = HERE if os.path.isdir(os.path.join(HERE, 'derivus')) else os.path.dirname(HERE)
sys.path.insert(0, REPO)

import numpy as np
import torch
import derivus as rf
from derivus.config import CustomJsonEncoder

WORLD = os.path.join(REPO, 'tests', 'fixtures', 'data', 'logvar2fj_world.json')
MARKET = json.load(open(WORLD))['MarketData']
BASE = MARKET['System Parameters']['Base_Date']['.Timestamp']
FACTOR, BLOCK = 'LogVar2FJModelParameters.INDEX_A', 'LogVar2FJModelPrices.INDEX_A'

print('derivus   ', os.path.dirname(rf.__file__))
print('device    ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('world     ', os.path.relpath(WORLD, REPO), '- base date', BASE)

derivus    C:\Users\Vretiel\PycharmProjects\derivus\derivus
device     NVIDIA GeForce RTX 3090
world      tests\fixtures\data\logvar2fj_world.json - base date 2024-06-28


## The deal

A synthetic autocall on the synthetic index of notebook 1, authored here so a reader can see
every field of it. Four fixings at roughly half-yearly intervals, a coupon ladder of 2, 4, 6 and
8 percent, an autocall threshold at the strike, and a terminal put struck at par knocking in
below 70 percent of it. No book, no counterparty, no trade: the only equity it names is
`INDEX_A`, which exists nowhere but the repository's own fixture.

In [2]:
FIXINGS = ['2024-12-27', '2025-06-27', '2025-12-26', '2026-06-26']
COUPONS = [0.02, 0.04, 0.06, 0.08]
DEAL = {'Object': 'QEDI_CustomAutoCallSwap', 'Reference': 'SYNTHETIC_AC', 'Tags': '', 'MtM': '',
        'Currency': 'EUR', 'Payoff_Currency': 'EUR', 'Equity': 'INDEX_A', 'Dividends': 'INDEX_A',
        'Discount_Rate': 'EUR', 'Equity_Volatility': 'INDEX_A.EUR', 'Buy_Sell': 'Buy',
        'Option_Type': 'Put', 'Strike_Price': 100.0, 'Expiry_Date': {'.Timestamp': FIXINGS[-1]},
        'Units': 10.0, 'Settlement_Style': 'Cash', 'Option_On_Forward': 'No',
        'Option_Style': 'European', 'Barrier': 0.7, 'Payoff_Type': 'Standard',
        'Price_Fixing': [[{'.Timestamp': d}, 0.0] for d in FIXINGS],
        'Autocall_Coupons': [[{'.Timestamp': d}, c] for d, c in zip(FIXINGS, COUPONS)],
        'Autocall_Thresholds': [[{'.Timestamp': d}, 1.0] for d in FIXINGS],
        'Barrier_Dates': [{'.Timestamp': d} for d in FIXINGS], 'Autocall_Floating': []}
print(json.dumps(DEAL, indent=1))

{
 "Object": "QEDI_CustomAutoCallSwap",
 "Reference": "SYNTHETIC_AC",
 "Tags": "",
 "MtM": "",
 "Currency": "EUR",
 "Payoff_Currency": "EUR",
 "Equity": "INDEX_A",
 "Dividends": "INDEX_A",
 "Discount_Rate": "EUR",
 "Equity_Volatility": "INDEX_A.EUR",
 "Buy_Sell": "Buy",
 "Option_Type": "Put",
 "Strike_Price": 100.0,
 "Expiry_Date": {
  ".Timestamp": "2026-06-26"
 },
 "Units": 10.0,
 "Settlement_Style": "Cash",
 "Option_On_Forward": "No",
 "Option_Style": "European",
 "Barrier": 0.7,
 "Payoff_Type": "Standard",
 "Price_Fixing": [
  [
   {
    ".Timestamp": "2024-12-27"
   },
   0.0
  ],
  [
   {
    ".Timestamp": "2025-06-27"
   },
   0.0
  ],
  [
   {
    ".Timestamp": "2025-12-26"
   },
   0.0
  ],
  [
   {
    ".Timestamp": "2026-06-26"
   },
   0.0
  ]
 ],
 "Autocall_Coupons": [
  [
   {
    ".Timestamp": "2024-12-27"
   },
   0.02
  ],
  [
   {
    ".Timestamp": "2025-06-27"
   },
   0.04
  ],
  [
   {
    ".Timestamp": "2025-12-26"
   },
   0.06
  ],
  [
   {
    ".Timestamp": "20

In [3]:
def run(declared, name, greeks='First', paths=32768):
    """Fit the world's ladder as the block declares it, then price the deal off the factor that
    fit just wrote - one document, one context, so the mark is the fit's own."""
    block = copy.deepcopy(MARKET['Market Prices'][BLOCK])
    block['instrument'].update(declared)
    job = {'Calc': {
        'Calculation': {'Object': 'BaseValuation', 'Base_Date': {'.Timestamp': BASE},
                        'Currency': 'USD', 'Greeks': greeks, 'MCMC_Simulations': paths,
                        'Random_Seed': 1},
        'MergeMarketData': {'MarketDataFile': WORLD, 'ExplicitMarketData': {
            'Market Prices': {BLOCK: block},
            'Valuation Configuration': {'QEDI_CustomAutoCallSwap': {
                'SpotModel': 'LogVar2FJ', 'Steps_Per_Year': 252.0, 'Internal_Step_Days': 1}}}},
        'Deals': {'Reference': 'notebook', 'Tag_Titles': '',
                  'Deals': {'Children': [{'Instrument': {'.Deal': DEAL}}]}}}}
    buf, root = io.StringIO(), logging.getLogger()
    saved, level = root.handlers[:], root.level
    root.handlers, root.level = [logging.StreamHandler(buf)], logging.INFO
    try:
        cx = rf.Context()
        cx.load_json((json.dumps(job, cls=CustomJsonEncoder), name + '.json'))
        cx.bootstrap()
        out = cx.run_job()[1]
    finally:
        root.handlers, root.level = saved, level
    return cx.current_cfg.params['Price Factors'].get(FACTOR), out, buf.getvalue()


def show(report, *markers):
    for line in report.splitlines():
        if any(marker in line for marker in markers):
            print(line.strip())

## The fit, and the reserve line it writes

The same ladder as notebook 1. What is read here is the reserve line: the tenor it was taken at,
the two derivatives, and the band.

In [4]:
started = time.time()
factor, out, report = run({'Max_Iterations': 60}, 'priors_on')
print('the fit and the mark took %.1f s\n' % (time.time() - started))
show(report, 'reserve line (')
print()
print('on the factor:  Skew_Gradient %s' % factor['Skew_Gradient'])
print('                Stickiness_Band %s vol points' % factor['Stickiness_Band'])

the fit and the mark took 14.7 s


on the factor:  Skew_Gradient 0.0165615607066,2.70871077411
                Stickiness_Band 0.5 vol points


The line says `REPORTED`. The forward block is off on this ladder — there are no forward-start
quotes to fit — so the window it was evaluated at is the ladder's own maturities and the
derivatives were not aimed at anything. That is the point: the reserve exists because the fit was
not given the rows, and the number is read off the fit it actually ran.

## The mark, and the reserve beside it

In [5]:
mtm = out['Results']['mtm']
print(mtm.to_string(index=False))
value = float(mtm.loc[mtm['Reference'] == DEAL['Reference'], 'Value'].iloc[0])
reported = float(mtm['Skew_Reserve'].dropna().iloc[0])

   Reference                  Object     Value  Skew_Reserve Parent Deal Currency  Ref_MTM
        root                    Root  0.000000      0.018143    NaN           NaN      NaN
SYNTHETIC_AC QEDI_CustomAutoCallSwap -1.085696      0.018143   root           EUR      0.0


### What the deal reads off the factor

Every parameter of the fitted factor, with the deal's derivative against it. This is the check
that the reserve is composed off a live sensitivity rather than a zero: the deal reads the
forward-variance curve and the leverage strongly, the residual's skew faintly and its tail almost
not at all — which is the same statement notebook 2 made from the other side.

In [6]:
greeks = out['Results']['Greeks_First']
column = [c for c in greeks.columns if c != 'Value'][0]
print('%-24s %16s %18s' % ('parameter', 'value', 'dPV/dparameter'))
for index in [i for i in greeks.index if str(i[0]).startswith(FACTOR)]:
    label = str(index[0])[len(FACTOR) + 1:] + ('[%gy]' % index[1] if index[1] else '')
    print('%-24s %16.8f %18.6e' % (label, float(greeks.loc[[index], 'Value'].iloc[0]),
                                   float(greeks.loc[[index], column].iloc[0])))

parameter                           value     dPV/dparameter
Alpha                         51.07062930       2.369966e-05
Beta                         -18.15203623       3.305091e-05
Kappa_L                        0.50000000      -1.680983e-02
Kappa_S                        6.00000000       4.677594e-03
Rho_L                         -0.40000000       2.725720e-01
Rho_S                         -0.81854713       9.829381e-02
Sigma_L                        1.00000000       5.957974e-02
Sigma_S                        1.93322357       3.290836e-04
Xi_Curve                       0.04236291      -2.400981e+00
Xi_Curve[0.0767123y]           0.04656315      -5.769714e+00
Xi_Curve[0.249315y]            0.05056608      -8.752558e+00
Xi_Curve[0.49863y]             0.05424403      -3.655841e+00
Xi_Curve[0.747945y]            0.05789319      -6.398604e+00


### Composed by hand

The engine's number, rebuilt from the factor's own gradient row and the deal's own two
derivatives, so the arithmetic is stated rather than trusted.

In [7]:
lever = [float(greeks.loc[[i for i in greeks.index
                           if str(i[0]) == '%s.%s' % (FACTOR, name)][-1], column])
         for name in ('Beta', 'Rho_S')]
row = [float(x) for x in str(factor['Skew_Gradient']).split(',')]
band = float(factor['Stickiness_Band'])

print('the deal      dPV/dBeta         %+18.8e' % lever[0])
print('              dPV/dRho_S        %+18.8e' % lever[1])
print('the factor    d(skew)/dBeta     %+18.8f vol points per unit' % row[0])
print('              d(skew)/dRho_S    %+18.8f vol points per unit' % row[1])
print('              band              %18.3f vol points' % band)
print()
per_vol_point = np.dot(lever, row) / np.dot(row, row)
hand = abs(per_vol_point) * band
print('dPV per vol point of forward skew, minimum norm  %+18.8f' % per_vol_point)
print('reserve composed here                            %18.8f' % hand)
print('reserve the engine reported                      %18.8f' % reported)
print('relative difference                              %18.2e' % (abs(reported - hand) / hand))
print()
print('the mark                                         %+18.8f' % value)
print('the reserve as a share of the mark               %17.2f%%' % (100 * reported / abs(value)))

the deal      dPV/dBeta            +3.30509133e-05
              dPV/dRho_S           +9.82938084e-02
the factor    d(skew)/dBeta            +0.01656156 vol points per unit
              d(skew)/dRho_S           +2.70871077 vol points per unit
              band                           0.500 vol points

dPV per vol point of forward skew, minimum norm         +0.03628676
reserve composed here                                    0.01814338
reserve the engine reported                              0.01814338
relative difference                                        0.00e+00

the mark                                                -1.08569640
the reserve as a share of the mark                            1.67%


### Per deal, or per portfolio

The reserve is composed from a gradient, and the calculation produces **one** gradient — the
netting set's, taken once on its total mark. So the number reported today sits on the row that
carries the portfolio's value, and a per-deal reserve would want a per-deal gradient this
calculation does not take. Where a per-deal column is present the cell below finds it and prints
it; where it is not, the portfolio number is what a report carries, and the per-deal column is
the one to read when it is there.

In [8]:
carried = mtm[mtm['Skew_Reserve'].notna()] if 'Skew_Reserve' in mtm.columns else mtm.iloc[:0]
print('%d of %d rows carry a reserve' % (len(carried), len(mtm)))
print(carried[['Reference', 'Object', 'Skew_Reserve']].to_string(index=False)
      if len(carried) else 'none')
per_deal = carried[carried['Reference'] == DEAL['Reference']]
print()
if len(per_deal):
    print('a PER-DEAL reserve is present in this engine and is the reported column: %.8f'
          % float(per_deal['Skew_Reserve'].iloc[0]))
else:
    print('no per-deal reserve in this engine. The number is the PORTFOLIO reserve, on the row')
    print('that carries the portfolio, composed from the netting set own single gradient. Where')
    print('a calculation produces a per-deal gradient the per-deal column is the reported one and')
    print('this notebook reads it instead; on a book of one deal the two coincide.')

2 of 2 rows carry a reserve
   Reference                  Object  Skew_Reserve
        root                    Root      0.018143
SYNTHETIC_AC QEDI_CustomAutoCallSwap      0.018143

a PER-DEAL reserve is present in this engine and is the reported column: 0.01814338


## What the priors are worth in the mark

`Model_Priors: Off` is the vanilla-only objective and the vanilla-only box: no leverage row, no
shape floor, no class prior on the residual. It is the same fifteen quotes fitted with nothing
holding the two directions the vanillas are flat in. The difference in the mark is what the
priors are worth.

In [9]:
started = time.time()
bare, bare_out, bare_report = run({'Max_Iterations': 60, 'Model_Priors': 'Off'}, 'priors_off')
print('the priors-off fit and mark took %.1f s\n' % (time.time() - started))
NAMES = ('Alpha', 'Beta', 'Rho_S', 'Sigma_S')
print('%-14s %14s %14s' % ('', 'priors on', 'priors off'))
for name in NAMES:
    print('%-14s %14.4f %14.4f' % (name, float(factor[name].array[0][1]),
                                   float(bare[name].array[0][1])))
for name in ('Rho_L', 'Sigma_L'):
    print('%-14s %14.4f %14.4f' % (name, float(factor[name]), float(bare[name])))
print()
for tag, written in (('priors on', factor), ('priors off', bare)):
    print('%-12s %s' % (tag, written['On_Guard'] or 'no guard'))

the priors-off fit and mark took 13.0 s

                    priors on     priors off
Alpha                 51.0706       194.8506
Beta                 -18.1520       144.5058
Rho_S                 -0.8185        -0.8186
Sigma_S                1.9332         2.0285
Rho_L                 -0.4000        -0.4000
Sigma_L                1.0000         1.0000

priors on    ON GUARD: c 0.170 at 0y within 0.05 of its C_Min floor 0.12
priors off   ON GUARD: |beta|/alpha 0.742 at 0y within 0.05 of 0.7746; c 0.170 at 0y within 0.05 of its C_Min floor 0.12


In [10]:
for tag, text in (('priors on', report), ('priors off', bare_report)):
    print('%-12s %s' % (tag, [ln.strip() for ln in text.splitlines()
                              if 'vol points unweighted' in ln][-1]))

priors on    RMSE 0.729 vol points unweighted over 15 quotes, 0.729 vega-weighted (the objective's own); the bootstrap's ATM misses +7.3e-11, -4.4e-16, +3.1e-13, +6.6e-12, +3.5e-11
priors off   RMSE 0.565 vol points unweighted over 15 quotes, 0.565 vega-weighted (the objective's own); the bootstrap's ATM misses +9.5e-11, +1.8e-15, +3.6e-13, +7.1e-12, +3.7e-11


In [11]:
bare_mtm = bare_out['Results']['mtm']
bare_value = float(bare_mtm.loc[bare_mtm['Reference'] == DEAL['Reference'], 'Value'].iloc[0])
bare_reserve = (float(bare_mtm['Skew_Reserve'].dropna().iloc[0])
                if 'Skew_Reserve' in bare_mtm.columns else float('nan'))
print('%-28s %16s %16s' % ('', 'priors on', 'priors off'))
print('%-28s %16.6f %16.6f' % ('the mark', value, bare_value))
print('%-28s %16.6f %16.6f' % ('the reserve', reported, bare_reserve))
print()
print('the mark moves %+0.6f, %+0.3f%% of the priors-on mark' % (
    bare_value - value, 100 * (bare_value - value) / abs(value)))
print('which is %.2f times the reserve the priors-on fit carries' % (
    abs(bare_value - value) / reported))

                                    priors on       priors off
the mark                            -1.085696        -1.094153
the reserve                          0.018143         0.004082

the mark moves -0.008457, -0.779% of the priors-on mark
which is 0.47 times the reserve the priors-on fit carries


### The reading

With the priors off the solver runs the residual pair to the edge of the box. `Alpha` leaves its
class prior for a number several times it, `Beta` changes sign, and the fit ends **on the
admissibility bound** — `|Beta|/Alpha` within 0.05 of its limit — which is the guard saying the
constraint is holding the answer and the data is not. It reads the fifteen quotes *better* for
it, by about a sixth of a vol point, because nothing is pulling against them any more. That is
precisely the flat direction: a large move in the residual pair that the vanillas barely notice.

And the same deal marks differently off the two factors, by about three quarters of a percent of
its value, and carries a reserve five times smaller off the second. The move is not noise — the
vanilla fits agree to a fraction of a vol point — it is the model's unidentified direction
arriving in a price.

So the priors are worth about that much in the mark, and the reserve is the number that says the
mark is uncertain by about that much. A validator should read the three together:

- **The mark** is what the fitted factor prices.
- **The identification table** says how much of that factor was data.
- **The reserve** is the mark's own statement of what the unquoted forward skew is worth,
  at a band a desk sets.

None of the three is meaningful without the other two, and the engine reports all three off the
same document.